In [1]:
from functools import partial
from time import perf_counter
from typing import NamedTuple
import threading

# If you are on Colab, uncomment the relevant curl line below.
# !curl -o lqrax.py https://raw.githubusercontent.com/MaxMSun/mcts-tutorial/refs/heads/main/lqrax.py

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import trimesh
import viser
from IPython.display import display, HTML
from IPython.utils.io import capture_output
from tqdm.auto import tqdm

from lqrax import LQR


In [2]:
def skew(v):
    x, y, z = v
    return jnp.array([[0., -z, y], [z, 0., -x], [-y, x, 0.]])


def quat_multiply(a, b):
    return jnp.concatenate((jnp.array([a[0] * b[0] - a[1:] @ b[1:]]),
                            a[0] * b[1:] + b[0] * a[1:] + jnp.cross(a[1:], b[1:])))


def quat_conjugate(q):
    return q * jnp.array([1., -1., -1., -1.])


def quat_exp(rotation):
    angle = jnp.sqrt(jnp.maximum(rotation @ rotation, 1e-12))
    return jnp.concatenate((jnp.cos(angle / 2)[None],
                            0.5 * jnp.sinc(angle / (2 * jnp.pi)) * rotation))


def quat_log(q):
    q = jnp.where(q[0] < 0, -q, q)
    norm = jnp.sqrt(jnp.maximum(q[1:] @ q[1:], 1e-12))
    return 2 * jnp.arctan2(norm, q[0]) / norm * q[1:]


def rotation_matrix(q):
    q = q / jnp.linalg.norm(q)
    w, v = q[0], q[1:]
    return (w * w - v @ v) * jnp.eye(3) + 2 * jnp.outer(v, v) + 2 * w * skew(v)


def state_plus(state, delta):
    return jnp.concatenate((state[:6] + delta[:6],
                            quat_multiply(state[6:10], quat_exp(delta[6:9])),
                            state[10:] + delta[9:]))


def state_error(state, reference):
    attitude = quat_log(quat_multiply(quat_conjugate(reference[6:10]), state[6:10]))
    return jnp.concatenate((state[:6] - reference[:6], attitude, state[10:] - reference[10:]))


def make_state(position, velocity=(0., 0., 0.)):
    return jnp.concatenate((jnp.asarray(position, dtype=jnp.float32),
                            jnp.asarray(velocity, dtype=jnp.float32),
                            jnp.array([1., 0., 0., 0., 0., 0., 0.])))


In [3]:
class Quadrotor:
    def __init__(self, bounds, cylinder_centers, cylinder_radius=0.55,
                 cylinder_height=2.8, dt=0.04):
        self.dt = dt
        self.mass, self.gravity = 1., 9.81
        self.arm, self.rotor_radius = 0.22, 0.10
        self.inertia = jnp.array([0.018, 0.022, 0.035])
        self.max_thrust, self.yaw_torque = 6., 0.025
        self.hover = jnp.full(4, self.mass * self.gravity / 4)
        a = self.arm / jnp.sqrt(2.)
        self.rotor_positions = jnp.array([[a, a, 0.], [-a, a, 0.],
                                          [-a, -a, 0.], [a, -a, 0.]])
        self.mixer = jnp.stack((self.rotor_positions[:, 1],
                               -self.rotor_positions[:, 0],
                               self.yaw_torque * jnp.array([1., -1., 1., -1.])))
        self.radius = self.arm + self.rotor_radius + 0.03
        self.bounds = jnp.asarray(bounds)
        self.cylinder_centers = jnp.asarray(cylinder_centers)
        self.cylinder_radius, self.cylinder_height = cylinder_radius, cylinder_height

    def dynamics(self, state, thrust):
        q, omega = state[6:10], state[10:]
        acceleration = rotation_matrix(q)[:, 2] * thrust.sum() / self.mass
        acceleration = acceleration - jnp.array([0., 0., self.gravity])
        q_dot = 0.5 * quat_multiply(q, jnp.concatenate((jnp.zeros(1), omega)))
        omega_dot = (self.mixer @ thrust - jnp.cross(omega, self.inertia * omega)) / self.inertia
        return jnp.concatenate((state[3:6], acceleration, q_dot, omega_dot))

    def rk4_step(self, state, thrust):
        k1 = self.dynamics(state, thrust)
        k2 = self.dynamics(state + self.dt * k1 / 2, thrust)
        k3 = self.dynamics(state + self.dt * k2 / 2, thrust)
        k4 = self.dynamics(state + self.dt * k3, thrust)
        result = state + self.dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6
        return result.at[6:10].set(result[6:10] / jnp.linalg.norm(result[6:10]))

    @partial(jax.jit, static_argnums=0)
    def trajectory(self, state, controls):
        def step(x, u):
            next_x = self.rk4_step(x, u)
            return next_x, next_x
        return jax.lax.scan(step, state, controls)[1]

    def linearize_continuous(self, state, thrust):
        R, omega = rotation_matrix(state[6:10]), state[10:]
        A = jnp.zeros((12, 12)).at[:3, 3:6].set(jnp.eye(3))
        A = A.at[3:6, 6:9].set(-R @ skew(jnp.array([0., 0., thrust.sum()])) / self.mass)
        A = A.at[6:9, 6:9].set(-skew(omega)).at[6:9, 9:].set(jnp.eye(3))
        angular = (skew(self.inertia * omega) - skew(omega) * self.inertia[None, :]) / self.inertia[:, None]
        A = A.at[9:, 9:].set(angular)
        B = jnp.zeros((12, 4)).at[3:6].set(jnp.repeat(R[:, 2:3] / self.mass, 4, axis=1))
        B = B.at[9:].set(self.mixer / self.inertia[:, None])
        return A, B

    def linearize_step(self, state, thrust):
        next_state = self.rk4_step(state, thrust)
        def perturbation(delta, control_delta):
            return state_error(self.rk4_step(state_plus(state, delta), thrust + control_delta), next_state)
        return jax.jacfwd(perturbation, argnums=(0, 1))(jnp.zeros(12), jnp.zeros(4))

    def collision(self, state, states):
        previous = jnp.roll(states[:, :3], 1, axis=0).at[0].set(state[:3])
        points = states[:, :3]
        outside = jnp.any((jnp.minimum(previous, points) < self.bounds[0] + self.radius)
                          | (jnp.maximum(previous, points) > self.bounds[1] - self.radius))

        # Intersect each swept segment with a conservatively inflated finite cylinder.
        direction = points - previous
        dz = direction[:, 2]
        divisor = jnp.where(jnp.abs(dz) > 1e-8, dz, 1.)
        t0 = (-self.radius - previous[:, 2]) / divisor
        t1 = (self.cylinder_height + self.radius - previous[:, 2]) / divisor
        lo = jnp.maximum(0., jnp.where(jnp.abs(dz) > 1e-8, jnp.minimum(t0, t1), 0.))
        hi = jnp.minimum(1., jnp.where(jnp.abs(dz) > 1e-8, jnp.maximum(t0, t1), 1.))
        z_inside = ((jnp.abs(dz) > 1e-8)
                    | ((previous[:, 2] >= -self.radius)
                       & (previous[:, 2] <= self.cylinder_height + self.radius)))
        offset = previous[:, None, :2] - self.cylinder_centers[None]
        dxy = direction[:, None, :2]
        closest_t = -jnp.sum(offset * dxy, axis=-1) / jnp.maximum(jnp.sum(dxy * dxy, axis=-1), 1e-12)
        closest_t = jnp.clip(closest_t, lo[:, None], hi[:, None])
        closest = offset + closest_t[..., None] * dxy
        hit = jnp.sum(closest * closest, axis=-1) <= (self.cylinder_radius + self.radius) ** 2
        return outside | jnp.any(hit & z_inside[:, None] & (lo <= hi)[:, None])

    def feasible(self, state, states, controls):
        up = jax.vmap(rotation_matrix)(states[:, 6:10])[:, 2, 2]
        return (~self.collision(state, states) & jnp.all(jnp.isfinite(states))
                & jnp.all(up > jnp.cos(1.0)) & jnp.all(jnp.abs(states[:, 10:]) < 7.)
                & jnp.all(controls >= 0.) & jnp.all(controls <= self.max_thrust))


In [4]:
class Candidates(NamedTuple):
    controls: jax.Array
    states: jax.Array
    valid: jax.Array


class SpectralExpansion:
    def __init__(self, system, steps=20, amplitude=3., trust_radius=1.2):
        self.system, self.steps = system, steps
        self.amplitude, self.trust_radius = amplitude, trust_radius
        self.state_scale = jnp.array([1., 1., 1., 2., 2., 2., 0.5, 0.5, 0.5, 2., 2., 2.])
        self.control_scale = system.max_thrust / 2
        self.Q = jnp.diag(jnp.array([3., 3., 3., 1., 1., 1., 6., 6., 3., 0.3, 0.3, 0.3]))
        self.R = jnp.eye(4) * 0.4
        self.lqr = LQR(system.dt, 12, 4, self.Q, self.R)
        A, B = system.linearize_continuous(make_state((0., 0., 0.)), system.hover)
        count = round(3.2 / system.dt)
        P = self.lqr.solve_riccati_backward(jnp.zeros((12, 12)),
                                           jnp.repeat(A[None], count, 0), jnp.repeat(B[None], count, 0))
        self.terminal_P = P[-1]
        self.hover_gain = jnp.linalg.solve(self.R, B.T @ self.terminal_P)

    def nominal(self, state):
        reference = make_state(state[:3])
        def step(x, _):
            u = jnp.clip(self.system.hover - self.hover_gain @ state_error(x, reference),
                         0., self.system.max_thrust)
            next_x = self.system.rk4_step(x, u)
            return next_x, (next_x, u)
        return jax.lax.scan(step, state, None, length=self.steps)[1]

    def endpoint_map(self, A, B):
        def step(product, pair):
            a, b = pair
            return product @ a, product @ b
        blocks = jax.lax.scan(step, jnp.eye(12), (A[::-1], B[::-1]))[1][::-1]
        C = blocks.transpose(1, 0, 2).reshape(12, -1)
        return C * self.control_scale / self.state_scale[:, None]

    def local_model(self, state):
        nominal, controls = self.nominal(state)
        previous = jnp.roll(nominal, 1, axis=0).at[0].set(state)
        A, B = jax.vmap(self.system.linearize_step)(previous, controls)
        C = self.endpoint_map(A, B)
        values, vectors = jnp.linalg.eigh(C @ C.T)
        values, vectors = jnp.maximum(values[::-1], 0.), vectors[:, ::-1]
        Ac, Bc = jax.vmap(self.system.linearize_continuous)(previous, controls)
        P = self.lqr.solve_riccati_backward(self.terminal_P, Ac[::-1], Bc[::-1])[::-1]
        gains = jax.vmap(lambda b, p: jnp.linalg.solve(self.R, b.T @ p))(Bc, P)
        return nominal, controls, A, B, C, values, vectors, gains

    @partial(jax.jit, static_argnums=0)
    def candidates(self, state):
        nominal, controls, A, B, C, values, vectors, gains = self.local_model(state)
        sigma = jnp.sqrt(jnp.maximum(values, 1e-12))
        directions = (C.T @ vectors / sigma[None, :]).T.reshape(12, self.steps, 4)
        magnitude = jnp.minimum(self.amplitude, self.trust_radius / sigma)
        deltas = self.control_scale * magnitude[:, None, None] * directions
        deltas = jnp.concatenate((jnp.zeros_like(deltas[:1]), deltas, -deltas))
        mode_valid = jnp.concatenate((jnp.ones(1, dtype=bool), values > values[0] * 1e-7,
                                     values > values[0] * 1e-7))
        previous_nominal = jnp.roll(nominal, 1, axis=0).at[0].set(state)

        def generate(delta):
            room = jnp.where(delta > 0., self.system.max_thrust - controls, controls)
            factor = jnp.minimum(1., jnp.min(room / jnp.maximum(jnp.abs(delta), 1e-8)))
            delta = delta * factor
            def perturb(z, pair):
                a, b, v = pair
                next_z = a @ z + b @ v
                return next_z, next_z
            z = jax.lax.scan(perturb, jnp.zeros(12), (A, B, delta))[1]
            previous_z = jnp.roll(z, 1, axis=0).at[0].set(jnp.zeros(12))
            references = jax.vmap(state_plus)(previous_nominal, previous_z)
            def track(x, data):
                ref, u, K = data
                applied = jnp.clip(u - K @ state_error(x, ref), 0., self.system.max_thrust)
                next_x = self.system.rk4_step(x, applied)
                return next_x, (next_x, applied)
            states, applied = jax.lax.scan(track, state, (references, controls + delta, gains))[1]
            return states, applied, self.system.feasible(state, states, applied)

        states, applied, valid = jax.vmap(generate)(deltas)
        return Candidates(applied, states, valid & mode_valid)


In [5]:
class SearchTree(NamedTuple):
    states: jax.Array
    parent: jax.Array
    incoming_action: jax.Array
    depth: jax.Array
    children: jax.Array
    controls: jax.Array
    successors: jax.Array
    rewards: jax.Array
    valid: jax.Array
    evaluated: jax.Array
    visits: jax.Array
    value_sum: jax.Array
    size: jax.Array


class SearchResult(NamedTuple):
    tree: SearchTree
    choice: jax.Array
    safe: jax.Array
    q_values: jax.Array


class SpectralMCTS:
    def __init__(self, expansion, goal, num_sim=512, max_depth=4, exploration=10.):
        self.expansion, self.system, self.goal = expansion, expansion.system, goal
        self.num_sim, self.max_depth, self.exploration = num_sim, max_depth, exploration
        self.num_actions = 25

    def cost(self, state):
        attitude = quat_log(state[6:10])
        return (jnp.sum((state[:3] - self.goal[:3]) ** 2)
                + 0.15 * jnp.sum(state[3:6] ** 2) + 0.3 * jnp.sum(attitude ** 2)
                + 0.02 * jnp.sum(state[10:] ** 2))

    def settled(self, state):
        return ((jnp.linalg.norm(state[:3] - self.goal[:3]) < 0.35)
                & (jnp.linalg.norm(state[3:6]) < 0.35)
                & (jnp.linalg.norm(quat_log(state[6:10])) < 0.15)
                & (jnp.linalg.norm(state[10:]) < 0.4))

    def leaf_value(self, state, depth):
        remaining_time = (self.max_depth - depth) * self.expansion.steps * self.system.dt
        return jnp.where(self.settled(state), 0., -(remaining_time + 2.) * self.cost(state))

    def empty_tree(self, state):
        n, a, h = self.num_sim + 1, self.num_actions, self.expansion.steps
        return SearchTree(
            jnp.zeros((n, 13)).at[0].set(state), jnp.full(n, -1, jnp.int32),
            jnp.zeros(n, jnp.int32), jnp.zeros(n, jnp.int32),
            jnp.full((n, a), -1, jnp.int32), jnp.zeros((n, a, h, 4)),
            jnp.zeros((n, a, 13)), jnp.zeros((n, a)), jnp.zeros((n, a), bool),
            jnp.zeros(n, bool), jnp.zeros((n, a), jnp.int32), jnp.zeros((n, a)),
            jnp.array(1, jnp.int32))

    def evaluate(self, tree, node):
        candidates = self.expansion.candidates(tree.states[node])
        costs = jax.vmap(jax.vmap(self.cost))(candidates.states)
        effort = 0.002 * jnp.sum((candidates.controls - self.system.hover) ** 2, axis=-1)
        rewards = -self.system.dt * jnp.sum(costs + effort, axis=1)
        return tree._replace(
            controls=tree.controls.at[node].set(candidates.controls),
            successors=tree.successors.at[node].set(candidates.states[:, -1]),
            rewards=tree.rewards.at[node].set(rewards), valid=tree.valid.at[node].set(candidates.valid),
            evaluated=tree.evaluated.at[node].set(True))

    def simulate(self, key, tree):
        path_nodes = jnp.zeros(self.max_depth, jnp.int32)
        path_actions = jnp.zeros(self.max_depth, jnp.int32)

        def condition(carry):
            tree, node, nodes, actions, length, active = carry
            return active & (length < self.max_depth) & ~self.settled(tree.states[node])

        def descend(carry):
            tree, node, nodes, actions, length, active = carry
            tree = jax.lax.cond(tree.evaluated[node], lambda t: t, lambda t: self.evaluate(t, node), tree)
            valid = tree.valid[node]
            untried = valid & (tree.children[node] < 0)
            has_new, has_valid = jnp.any(untried), jnp.any(valid)
            counts = tree.visits[node]
            q = tree.value_sum[node] / jnp.maximum(counts, 1)
            bonus = self.exploration * jnp.sqrt(counts.sum() + 1.) / (1. + counts)
            selected = jnp.argmax(jnp.where(valid, q + bonus, -jnp.inf))
            random_scores = jax.random.uniform(jax.random.fold_in(key, length), (self.num_actions,))
            new_action = jnp.argmax(jnp.where(untried, random_scores, -jnp.inf))
            action = jnp.where(has_new, new_action, selected)

            def add(t):
                child = t.size
                return t._replace(
                    states=t.states.at[child].set(t.successors[node, action]),
                    parent=t.parent.at[child].set(node), incoming_action=t.incoming_action.at[child].set(action),
                    depth=t.depth.at[child].set(t.depth[node] + 1),
                    children=t.children.at[node, action].set(child), size=child + 1)
            tree = jax.lax.cond(has_new, add, lambda t: t, tree)
            child = jnp.where(has_valid, tree.children[node, action], node)
            nodes = nodes.at[length].set(node)
            actions = actions.at[length].set(action)
            return tree, child, nodes, actions, length + has_valid.astype(jnp.int32), has_valid & ~has_new

        initial = (tree, jnp.array(0, jnp.int32), path_nodes, path_actions, jnp.array(0, jnp.int32), jnp.array(True))
        tree, node, nodes, actions, length, _ = jax.lax.while_loop(condition, descend, initial)
        dead = tree.evaluated[node] & ~jnp.any(tree.valid[node])
        value = jnp.where(dead, -1000. - 20. * self.cost(tree.states[node]), self.leaf_value(tree.states[node], tree.depth[node]))

        def backup(carry, index):
            tree, value = carry
            parent, action = nodes[index], actions[index]
            active = index < length
            value = jnp.where(active, tree.rewards[parent, action] + value, value)
            tree = tree._replace(
                visits=tree.visits.at[parent, action].add(active.astype(jnp.int32)),
                value_sum=tree.value_sum.at[parent, action].add(jnp.where(active, value, 0.)))
            return (tree, value), None
        return jax.lax.scan(backup, (tree, value), jnp.arange(self.max_depth - 1, -1, -1))[0][0]

    @partial(jax.jit, static_argnums=0)
    def search(self, key, state):
        tree = self.empty_tree(state)
        tree = jax.lax.fori_loop(0, self.num_sim,
                                lambda i, t: self.simulate(jax.random.fold_in(key, i), t), tree)
        counts = tree.visits[0]
        q = jnp.where(counts > 0, tree.value_sum[0] / jnp.maximum(counts, 1), jnp.nan)
        choice = jnp.argmax(jnp.where(tree.valid[0], counts, -1))
        return SearchResult(tree, choice, jnp.any(tree.valid[0]), q)


In [6]:
def simulate_flight(search, start, key, total_time=40., execution_steps=None, capture_tree=True):
    system, expansion = search.system, search.expansion
    execution_steps = expansion.steps // 2 if execution_steps is None else execution_steps
    if not 1 <= execution_steps <= expansion.steps:
        raise ValueError('execution_steps must be between 1 and the action horizon')
    total_steps = round(total_time / system.dt)
    state = start
    history = dict(states=[np.asarray(start)], controls=[], trees=[], plan_index=[0],
                   planning_times=[], decision_steps=[], status='Time limit reached')
    decision = 0
    with tqdm(total=total_steps, desc='Simulating', unit='step', leave=False) as progress:
        while len(history['controls']) < total_steps:
            if bool(search.settled(state)):
                history['status'] = 'Reached goal'
                break
            begin = perf_counter()
            result = search.search(jax.random.fold_in(key, decision), state)
            jax.block_until_ready(result)
            history['planning_times'].append(perf_counter() - begin)
            if not bool(result.safe):
                history['status'] = 'No collision-free action'
                break
            count = min(execution_steps, total_steps - len(history['controls']))
            controls = result.tree.controls[0, result.choice, :count]
            states = system.trajectory(state, controls)
            if not bool(system.feasible(state, states, controls)):
                history['status'] = 'Execution validation failed'
                break
            if capture_tree:
                history['trees'].append(tree_snapshot(result, system))
            history['decision_steps'].append(len(history['controls']))
            history['plan_index'][-1] = decision
            for next_state, control in zip(states, controls):
                state = next_state
                history['states'].append(np.asarray(state))
                history['controls'].append(np.asarray(control))
                history['plan_index'].append(decision)
                progress.update(1)
                if bool(search.settled(state)):
                    history['status'] = 'Reached goal'
                    break
            if history['status'] == 'Reached goal':
                break
            decision += 1
    for name in ('states', 'controls', 'plan_index', 'planning_times', 'decision_steps'):
        history[name] = np.asarray(history[name])
    return history


In [7]:
system = Quadrotor(
    bounds=[[0., 0., 0.], [10., 8., 5.]],
    cylinder_centers=[[2.8, 2.0], [3., 4.8], [5., 3.8], [5., 6.], [7., 5.4], [7., 2.5]],
    cylinder_radius=0.55,
    cylinder_height=2.8,
    dt=0.04,
)
start = make_state((1., 1., 1.))
goal = make_state((9., 7., 2.2))
expansion = SpectralExpansion(system, steps=20, amplitude=3., trust_radius=1.2)
search = SpectralMCTS(expansion, goal, num_sim=512, max_depth=4, exploration=10.)
execution_steps = expansion.steps // 2
total_time = 40.
key = jax.random.PRNGKey(0)


In [8]:
# Visualization helpers; the planner above has no dependency on Viser.
def drone_meshes(system):
    body = trimesh.creation.box(extents=(0.20, 0.13, 0.09))
    parts = [body]
    for rotor in np.asarray(system.rotor_positions):
        arm = trimesh.creation.box(extents=(system.arm, 0.035, 0.025))
        angle = np.arctan2(rotor[1], rotor[0])
        arm.apply_transform(trimesh.transformations.rotation_matrix(angle, (0., 0., 1.)))
        arm.apply_translation(rotor / 2)
        parts.append(arm)
    frame = trimesh.util.concatenate(parts)
    rotors = []
    for position in np.asarray(system.rotor_positions):
        rim = trimesh.creation.annulus(r_min=system.rotor_radius - 0.012,
                                       r_max=system.rotor_radius, height=0.014, sections=20)
        rim.apply_translation(position + np.array([0., 0., 0.025]))
        hub = trimesh.creation.cylinder(radius=0.025, height=0.06, sections=12)
        hub.apply_translation(position)
        blade = trimesh.creation.box(extents=(system.rotor_radius * 1.8, 0.018, 0.006))
        blade.apply_translation(position + np.array([0., 0., 0.03]))
        rotors.extend((rim, hub, blade))
    rotors = trimesh.util.concatenate(rotors)
    return frame, rotors, trimesh.util.concatenate((frame, rotors))


@partial(jax.jit, static_argnums=0)
def edge_trajectories(system, states, controls):
    return jax.vmap(system.trajectory)(states, controls)


def tree_snapshot(result, system, max_edges=160, max_leaves=60):
    tree = result.tree
    size = int(tree.size)
    parent = np.asarray(tree.parent[:size])
    action = np.asarray(tree.incoming_action[:size])
    children = np.asarray(tree.children[:size])
    visits = np.asarray(tree.visits[:size])
    value_sum = np.asarray(tree.value_sum[:size])
    states = np.asarray(tree.states[:size])
    nodes = np.arange(1, size)
    counts = visits[parent[nodes], action[nodes]]
    order = np.lexsort((np.asarray(tree.depth[1:size]), -counts))
    nodes = nodes[order[:max_edges]]

    # Pad only the visualization batch so changing tree sizes do not recompile JAX.
    padded = np.pad(nodes, (0, max_edges - len(nodes)), constant_values=0)
    parents = np.maximum(parent[padded], 0)
    controls = tree.controls[parents, action[padded]]
    trajectories = np.asarray(edge_trajectories(system, tree.states[parents], controls))[:len(nodes)]
    origins = states[parent[nodes]]
    previous = np.roll(trajectories[:, :, :3], 1, axis=1)
    previous[:, 0] = origins[:, :3]
    segments = np.stack((previous, trajectories[:, :, :3]), axis=2)
    q = value_sum[parent[nodes], action[nodes]] / np.maximum(visits[parent[nodes], action[nodes]], 1)
    low, high = np.percentile(q, (5, 95)) if len(q) else (0., 1.)
    colors = plt.colormaps['viridis'](np.clip((q - low) / max(high - low, 1e-6), 0., 1.))[:, :3]
    colors = np.asarray((0.55 + 0.45 * colors) * 255, dtype=np.uint8)
    colors = np.repeat(colors[:, None, None, :], trajectories.shape[1], axis=1)
    colors = np.repeat(colors, 2, axis=2)
    leaves = nodes[np.all(children[nodes] < 0, axis=1)][:max_leaves]
    selected_controls = tree.controls[0, result.choice]
    selected = np.asarray(system.trajectory(tree.states[0], selected_controls))
    return dict(segments=segments.reshape(-1, 2, 3), colors=colors.reshape(-1, 2, 3),
                leaves=states[leaves], selected=selected, root=states[0])


class FlightViewer:
    def __init__(self, system, start, goal):
        self.system = system
        self.server = viser.ViserServer(host='127.0.0.1', verbose=False)
        self.server.scene.set_up_direction('+z')
        self.server.initial_camera.position = (13., -11., 12.)
        self.server.initial_camera.look_at = (5., 4., 1.8)
        self.server.initial_camera.up = (0., 0., 1.)
        self.server.initial_camera.fov = np.deg2rad(40.)
        self.frame_mesh, self.rotor_mesh, self.ghost_mesh = drone_meshes(system)
        self.tree_root = self.server.scene.add_frame('/search', show_axes=False)
        self.leaf_root = self.server.scene.add_frame('/search/leaves', show_axes=False)
        self.server.scene.add_grid('/ground', width=10., height=8., plane='xy',
                                   position=(5., 4., 0.), plane_color=(247, 247, 247), plane_opacity=1.)
        for i, center in enumerate(np.asarray(system.cylinder_centers)):
            mesh = trimesh.creation.cylinder(radius=system.cylinder_radius,
                                             height=system.cylinder_height, sections=40)
            self.server.scene.add_mesh_simple(f'/obstacles/{i}', mesh.vertices, mesh.faces,
                color=(155, 162, 168), position=(*center, system.cylinder_height / 2))
        for name, state, color in [('start', start, (95, 95, 95)), ('goal', goal, (30, 160, 95))]:
            self.server.scene.add_icosphere('/' + name, radius=0.10, color=color, position=np.asarray(state[:3]))
        self.drone = self.server.scene.add_frame('/drone', show_axes=False,
                                                 position=np.asarray(start[:3]), wxyz=np.asarray(start[6:10]))
        self.server.scene.add_mesh_simple('/drone/body', self.frame_mesh.vertices, self.frame_mesh.faces,
                                          color=(35, 38, 42))
        self.server.scene.add_mesh_simple('/drone/rotors', self.rotor_mesh.vertices, self.rotor_mesh.faces,
                                          color=(175, 181, 187))
        nose = trimesh.creation.box(extents=(0.035, 0.11, 0.045))
        self.server.scene.add_mesh_simple('/drone/front', nose.vertices, nose.faces,
                                          color=(240, 125, 35), position=(0.10, 0., 0.02))
        self.show_tree = self.server.gui.add_checkbox('Search tree', True)
        self.show_leaves = self.server.gui.add_checkbox('Leaf drones', True)
        self.show_tree.on_update(lambda _: setattr(self.tree_root, 'visible', self.show_tree.value))
        self.show_leaves.on_update(lambda _: setattr(self.leaf_root, 'visible', self.show_leaves.value))
        self.edges = self.server.scene.add_line_segments('/search/edges', np.zeros((1, 2, 3)),
            (180, 180, 180), thickness=1., thickness_units='screen')
        self.leaf_drones = self.server.scene.add_batched_meshes_simple('/search/leaves/drones',
            self.ghost_mesh.vertices, self.ghost_mesh.faces,
            batched_wxyzs=np.tile([1., 0., 0., 0.], (60, 1)), batched_positions=np.zeros((60, 3)),
            batched_scales=np.zeros(60), batched_colors=(110, 115, 120), opacity=0.14,
            lod='off', cast_shadow=False)
        self.selected_path = self.server.scene.add_line_segments('/search/selected', np.zeros((1, 2, 3)),
            (230, 125, 30), thickness=3., thickness_units='screen')
        self.endpoint = self.server.scene.add_mesh_simple('/search/endpoint',
            self.ghost_mesh.vertices, self.ghost_mesh.faces,
            color=(230, 125, 30), opacity=0.45, cast_shadow=False, visible=False)
        self.stop_event = threading.Event()
        self.playback_thread = None
        self.current_plan = -1

    def close(self):
        self.stop_event.set()
        if self.playback_thread is not None:
            self.playback_thread.join(timeout=1.)
        self.server.stop()

    def set_tree(self, snapshot):
        self.edges.points = snapshot['segments']
        self.edges.colors = snapshot['colors']
        leaves = snapshot['leaves']
        count = len(leaves)
        positions = np.zeros((60, 3), dtype=np.float32)
        orientations = np.tile(np.array([1., 0., 0., 0.], dtype=np.float32), (60, 1))
        scales = np.zeros(60, dtype=np.float32)
        positions[:count], orientations[:count], scales[:count] = leaves[:, :3], leaves[:, 6:10], 0.8
        self.leaf_drones.batched_positions = positions
        self.leaf_drones.batched_wxyzs = orientations
        self.leaf_drones.batched_scales = scales
        selected = snapshot['selected']
        previous = np.roll(selected[:, :3], 1, axis=0)
        previous[0] = snapshot['root'][:3]
        self.selected_path.points = np.stack((previous, selected[:, :3]), axis=1)
        self.endpoint.position, self.endpoint.wxyz = selected[-1, :3], selected[-1, 6:10]
        self.endpoint.visible = True

    def load(self, history):
        if self.playback_thread is not None:
            self.stop_event.set()
            self.playback_thread.join(timeout=1.)
            for control in (self.play, self.speed, self.frame):
                control.remove()
            self.stop_event = threading.Event()
        self.history = history
        self.current_plan = -1
        states = history['states']
        self.server.scene.add_line_segments('/path', np.stack((states[:-1, :3], states[1:, :3]), axis=1),
                                            (35, 40, 45), thickness=2., thickness_units='screen')
        self.play = self.server.gui.add_checkbox('Play', False)
        self.speed = self.server.gui.add_slider('Playback speed', min=0.25, max=2., step=0.25, initial_value=1.)
        self.frame = self.server.gui.add_slider('Frame', min=0, max=len(states) - 1, step=1, initial_value=0)
        self.frame.on_update(lambda _: self.set_frame(self.frame.value))
        self.set_frame(0)

        def playback():
            while not self.stop_event.is_set():
                if self.play.value:
                    self.frame.value = (self.frame.value + 1) % len(states)
                self.stop_event.wait(self.system.dt / self.speed.value)
        self.playback_thread = threading.Thread(target=playback, daemon=True)
        self.playback_thread.start()

    def set_frame(self, index):
        state = self.history['states'][index]
        plan = self.history['plan_index'][index]
        with self.server.atomic():
            self.drone.position, self.drone.wxyz = state[:3], state[6:10]
            if plan != self.current_plan:
                self.set_tree(self.history['trees'][plan])
                self.current_plan = plan

    def recording(self):
        self.play.value = False
        self.current_plan = -1
        self.set_frame(0)
        recording = self.server.get_scene_serializer()
        for i in tqdm(range(1, len(self.history['states'])), desc='Rendering animation', unit='frame', leave=False):
            recording.insert_sleep(self.system.dt)
            self.set_frame(i)
        return recording


In [9]:
_ = jax.block_until_ready(search.search(key, start))
history = simulate_flight(search, start, key, total_time=total_time, execution_steps=execution_steps)


Simulating:   0%|          | 0/1000 [00:00<?, ?step/s]

In [10]:
with capture_output():
    if 'viewer' in globals():
        viewer.close()
    viewer = FlightViewer(system, start, goal)

if len(history['trees']) and len(history['states']) > 1:
    viewer.load(history)
    recording = viewer.recording()
    recording.show(height=750)
    viewer.set_frame(0)
    display(HTML(f'<a href="http://localhost:{viewer.server.get_port()}" target="_blank">'
                 'Live viewer: playback speed, search tree, and leaf drone controls</a>'))
else:
    viewer.server.scene.show(height=750)


Rendering animation:   0%|          | 0/759 [00:00<?, ?frame/s]